# STEP04 — MOTH EXP047-B risk cache +20k

Продолжаем **ровно принятую линию EXP047-B** для `ilPloInte3.2`.

Вход:
- moth EXP045-equivalent @50k;
- frozen checkpoint SHA-256 проверяется;
- validation в mining **не участвует**.

Этапы:
1. frozen EXP045 @50k делает exhaustive inference **только на train**;
2. train negatives ранжируются по `sigmoid(logit)`, округлённому до 5 знаков;
3. строятся те же четыре EXP047 danger strata: top 1%, 1–5%, 5–15%, bottom 85%;
4. применяются те же risk multipliers;
5. EXP047-B продолжает обучение ещё 20k шагов;
6. только после step 70k делается один полный natural validation scan.

Мы **не выбираем checkpoint по validation** и не меняем рецепт под моль.


In [1]:
from pathlib import Path
from datetime import date
import copy
import gc
import hashlib
import json
import math
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch_directml
from IPython.display import display

EXPECTED_PYTHON = Path(
    r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe"
).resolve()
ACTUAL_PYTHON = Path(sys.executable).resolve()
assert ACTUAL_PYTHON == EXPECTED_PYTHON, (ACTUAL_PYTHON, EXPECTED_PYTHON)

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (current, *current.parents)
    if (p / "README.md").is_file()
    and (p / "src/ml_project").is_dir()
    and (p / "data/raw/unpacked").is_dir()
)

AGENTS_PATH = next(
    (
        p for p in (
            PROJECT_ROOT / "AGENTS.md",
            PROJECT_ROOT.parent / "AGENTS.md",
        )
        if p.is_file()
    ),
    None,
)
assert AGENTS_PATH is not None

agents_sha = hashlib.sha256(AGENTS_PATH.read_bytes()).hexdigest()
EXPECTED_AGENTS_SHA = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
assert agents_sha == EXPECTED_AGENTS_SHA, (agents_sha, EXPECTED_AGENTS_SHA)

LOADER_PATH = PROJECT_ROOT / "src/ml_project/epic_data.py"
loader_sha = hashlib.sha256(LOADER_PATH.read_bytes()).hexdigest()
EXPECTED_LOADER_SHA = "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468"
assert loader_sha == EXPECTED_LOADER_SHA, (loader_sha, EXPECTED_LOADER_SHA)

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    evaluate_presence_intensity_model,
    forward_both_strands_multitask,
)
from ml_project.epic_data import SplitConfig, load_split

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def atomic_json(path, payload):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(tmp, path)

def atomic_torch_save(path, payload):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

print("Python:", ACTUAL_PYTHON)
print("Project:", PROJECT_ROOT)
print("GPU:", torch_directml.device_name(0))
print("AGENTS SHA:", agents_sha)
print("Loader SHA:", loader_sha)


Python: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
Project: D:\Projects\EPIC\EPIC
GPU: AMD Radeon RX 6750 GRE 12GB 
AGENTS SHA: bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3
Loader SHA: 37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468


## 1. Frozen inputs


In [2]:
ASSEMBLY = "ilPloInte3.2"

SPLIT_CONFIG = SplitConfig(
    assembly=ASSEMBLY,
    split_version="contig_holdout_v1",
    seed=42,
    validation_contig_fraction=0.25,
)

split = load_split(PROJECT_ROOT, SPLIT_CONFIG, verify_hashes=True)
sequences, _ = load_baseline_sequences(split)

BASE_RUN = (
    PROJECT_ROOT
    / "artifacts/experiments"
    / "SUBMIT_ilPloInte3p2_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029"
    / "run_001"
)
BASE_CHECKPOINT = BASE_RUN / "model_step_50000.pt"
BASE_METRICS = BASE_RUN / "validation_metrics_report.csv"

assert BASE_CHECKPOINT.is_file(), BASE_CHECKPOINT
assert BASE_METRICS.is_file(), BASE_METRICS

base_sha = sha256_file(BASE_CHECKPOINT)
EXPECTED_BASE_SHA = "01dd1761937dbe15ce334689db27221cc0801514a2bf9a7e63ee81ec9d521048"
assert base_sha == EXPECTED_BASE_SHA, (
    "Unexpected moth EXP045 checkpoint. "
    f"Current={base_sha}, expected={EXPECTED_BASE_SHA}"
)

base = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)
assert base["assembly"] == ASSEMBLY
assert int(base["step"]) == 50_000
assert base["architecture"] == "rf1029_width512"
assert int(base["model_config"]["channels"]) == 512
assert int(base["model_config"]["theoretical_rf"]) == 1029

profile_config = ProfileWindowConfig(**base["profile_config"])
assert profile_config.target_length == 1024
assert profile_config.context_flank == 130
assert profile_config.positive_branch_fraction == 0.5

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
train_tiles = train_generator.tile_index.reset_index(drop=True).copy()

tile_ids = train_tiles["tile_id"].to_numpy(np.int64)
assert np.array_equal(tile_ids, np.arange(len(train_tiles), dtype=np.int64))

assert int(train_generator.total_positive) == 581_052
assert int(train_generator.total_negative) == 328_535_026

print("Base checkpoint:", BASE_CHECKPOINT)
print("SHA256:", base_sha)
print("Train tiles:", f"{len(train_tiles):,}")
print("Train positives:", f"{train_generator.total_positive:,}")
print("Train negatives:", f"{train_generator.total_negative:,}")
print("Train position-strand:", f"{train_generator.total_positive + train_generator.total_negative:,}")


Base checkpoint: D:\Projects\EPIC\EPIC\artifacts\experiments\SUBMIT_ilPloInte3p2_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\model_step_50000.pt
SHA256: 01dd1761937dbe15ce334689db27221cc0801514a2bf9a7e63ee81ec9d521048
Train tiles: 168,611
Train positives: 581,052
Train negatives: 328,535,026
Train position-strand: 329,116,078


## 2. EXP047-B frozen constants

Эти значения переносим из принятого EXP047-B без изменения.


In [3]:
EXPERIMENT_ID = "SUBMIT-MOTH-EXP047B"
EXPERIMENT_SLUG = "SUBMIT_ilPloInte3p2_EXP047B_RISK_CONTINUATION"

BASE_STEP = 50_000
CONTINUATION_STEPS = 20_000
BATCH_SIZE = 4

WARMUP_STEPS = 500
START_LR = 3e-5
PEAK_LR = 1e-4
END_LR = 1e-5

INTENSITY_LOSS_WEIGHT = 0.1
TARGET_NEGATIVE_STRATUM_FRACTION = 0.25
SCORE_LEVELS = 100_001

LOG_EVERY = 250
LATEST_EVERY = 500
SNAPSHOT_STEPS = {2_000, 5_000, 10_000, 15_000, 20_000}

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts/experiments"
    / EXPERIMENT_SLUG
    / "run_001"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

RISK_DIR = RUN_DIR / "risk_strata_cache"
RISK_DIR.mkdir(parents=True, exist_ok=True)

SCORE_BIN_PATH = RISK_DIR / "train_exp045_probability_bin_u32.dat"
SCORE_META_PATH = RISK_DIR / "train_exp045_probability_bin_manifest.json"
HISTOGRAM_PATH = RISK_DIR / "train_negative_score_histogram.npy"
RISK_CACHE = RISK_DIR / "train_negative_stratum_u8.dat"
RISK_MANIFEST = RISK_DIR / "risk_strata_manifest.json"

BRANCH_DIR = RUN_DIR / "branch_B"
BRANCH_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CHECKPOINT = BRANCH_DIR / "branch_B_final.pt"
LATEST_CHECKPOINT = BRANCH_DIR / "latest.pt"
TRAINING_HISTORY = BRANCH_DIR / "training_history.csv"
VALIDATION_METRICS = BRANCH_DIR / "validation_metrics_report.csv"
RESULT_SUMMARY = RUN_DIR / "result_summary.json"

SCORE_SHAPE = (
    len(train_tiles),
    2,
    profile_config.target_length,
)

score_bytes = int(np.prod(SCORE_SHAPE, dtype=np.int64)) * np.dtype(np.uint32).itemsize
risk_bytes = int(np.prod(SCORE_SHAPE, dtype=np.int64)) * np.dtype(np.uint8).itemsize

print("Risk run:", RUN_DIR)
print("Score-bin cache size:", f"{score_bytes / 1024**3:.2f} GiB")
print("Risk cache size:", f"{risk_bytes / 1024**3:.2f} GiB")

def planned_lr(local_step):
    if not 1 <= local_step <= CONTINUATION_STEPS:
        raise ValueError(local_step)
    if local_step <= WARMUP_STEPS:
        fraction = local_step / WARMUP_STEPS
        return START_LR + fraction * (PEAK_LR - START_LR)
    fraction = (
        (local_step - WARMUP_STEPS)
        / (CONTINUATION_STEPS - WARMUP_STEPS)
    )
    return END_LR + 0.5 * (PEAK_LR - END_LR) * (
        1.0 + math.cos(math.pi * fraction)
    )

print(
    "LR:",
    planned_lr(1),
    planned_lr(500),
    planned_lr(20_000),
)
assert math.isclose(planned_lr(500), PEAK_LR)
assert math.isclose(planned_lr(20_000), END_LR)


Risk run: D:\Projects\EPIC\EPIC\artifacts\experiments\SUBMIT_ilPloInte3p2_EXP047B_RISK_CONTINUATION\run_001
Score-bin cache size: 1.29 GiB
Risk cache size: 0.32 GiB
LR: 3.014e-05 0.0001 1e-05


## 3. Exhaustive frozen EXP045 inference on TRAIN only

Здесь **нет validation**.

Для каждого train tile сохраняем score bin:

`round(sigmoid(EXP045_logit) × 100000)`

Minus-strand разворачивается обратно в genomic left→right до записи.

Файл resumable: если процесс оборвётся, повторный запуск продолжится с последнего
зафиксированного tile.


In [4]:
def sigmoid_numpy(logits):
    logits = np.asarray(logits, dtype=np.float64)
    out = np.empty_like(logits)
    positive = logits >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-logits[positive]))
    exp_x = np.exp(logits[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out

def score_bins(logits):
    return np.rint(
        np.clip(sigmoid_numpy(logits), 0.0, 1.0) * 100_000
    ).astype(np.uint32)

expected_score_bytes = int(np.prod(SCORE_SHAPE, dtype=np.int64)) * 4

if SCORE_META_PATH.is_file():
    score_meta = json.loads(SCORE_META_PATH.read_text(encoding="utf-8"))
    assert score_meta["source_checkpoint_sha256"] == base_sha
    assert tuple(score_meta["shape"]) == SCORE_SHAPE
    assert score_meta["dtype"] == "uint32"
    assert SCORE_BIN_PATH.is_file()
    assert SCORE_BIN_PATH.stat().st_size == expected_score_bytes
    completed_tiles = int(score_meta["completed_tiles"])
    assert 0 <= completed_tiles <= len(train_tiles)
    score_map = np.memmap(
        SCORE_BIN_PATH,
        mode="r+",
        dtype=np.uint32,
        shape=SCORE_SHAPE,
    )
    print("Resume train score cache at tile:", completed_tiles)
else:
    if SCORE_BIN_PATH.exists():
        raise RuntimeError(
            "Score cache exists without manifest; refusing ambiguous resume."
        )
    score_map = np.memmap(
        SCORE_BIN_PATH,
        mode="w+",
        dtype=np.uint32,
        shape=SCORE_SHAPE,
    )
    completed_tiles = 0
    score_meta = {
        "complete": False,
        "assembly": ASSEMBLY,
        "source_checkpoint": str(BASE_CHECKPOINT),
        "source_checkpoint_sha256": base_sha,
        "shape": list(SCORE_SHAPE),
        "dtype": "uint32",
        "score_definition": "round(sigmoid(EXP045 presence logit) * 100000)",
        "coordinate_order": "tile_id, genomic strand(+,-), genomic left-to-right base",
        "completed_tiles": 0,
    }
    atomic_json(SCORE_META_PATH, score_meta)

if completed_tiles < len(train_tiles):
    device = torch_directml.device()

    score_model = create_cnn_presence_intensity(channels=512)
    score_model.load_state_dict(base["model_state_dict"])
    score_model = score_model.to(device)
    score_model.eval()

    EVAL_BATCH_SIZE = 1
    started = time.perf_counter()
    initial_completed = completed_tiles

    with torch.no_grad():
        for start in range(completed_tiles, len(train_tiles), EVAL_BATCH_SIZE):
            end = min(start + EVAL_BATCH_SIZE, len(train_tiles))
            selected = train_tiles.iloc[start:end]
            batch = train_generator.make_batch(selected)

            batch_tile_ids = batch.tile_id.numpy().astype(np.int64, copy=False)
            expected_ids = selected["tile_id"].to_numpy(np.int64)
            assert np.array_equal(batch_tile_ids, expected_ids)

            x_gpu = batch.x_both_strands.to(device)

            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                plus_gpu, minus_reverse_gpu, _, _ = (
                    forward_both_strands_multitask(
                        score_model,
                        x_gpu,
                        batch_size=len(selected),
                        context_flank=profile_config.context_flank,
                        target_length=profile_config.target_length,
                    )
                )

            fallbacks = [
                str(item.message)
                for item in caught
                if "fall back" in str(item.message).lower()
            ]
            if fallbacks:
                raise RuntimeError(
                    "DirectML CPU fallback during train scoring: "
                    + " | ".join(fallbacks)
                )

            plus_logit = (
                plus_gpu[:, 0, :]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32, copy=True)
            )
            minus_logit = (
                minus_reverse_gpu[:, 0, :]
                .detach()
                .cpu()
                .numpy()[:, ::-1]
                .astype(np.float32, copy=True)
            )

            score_map[batch_tile_ids, 0, :] = score_bins(plus_logit)
            score_map[batch_tile_ids, 1, :] = score_bins(minus_logit)

            completed_tiles = end

            if (
                completed_tiles % 1000 < EVAL_BATCH_SIZE
                or completed_tiles == len(train_tiles)
            ):
                score_map.flush()
                elapsed = time.perf_counter() - started
                done_now = completed_tiles - initial_completed
                tiles_per_second = done_now / max(elapsed, 1e-9)
                remaining = len(train_tiles) - completed_tiles

                score_meta["completed_tiles"] = completed_tiles
                score_meta["complete"] = (
                    completed_tiles == len(train_tiles)
                )
                score_meta["last_update_epoch_seconds"] = time.time()
                atomic_json(SCORE_META_PATH, score_meta)

                print(
                    f"train score tiles "
                    f"{completed_tiles:,}/{len(train_tiles):,} "
                    f"({100*completed_tiles/len(train_tiles):.2f}%) "
                    f"tiles/s={tiles_per_second:.2f} "
                    f"ETA={remaining/max(tiles_per_second,1e-9)/3600:.2f}h",
                    flush=True,
                )

            del (
                batch, x_gpu,
                plus_gpu, minus_reverse_gpu,
                plus_logit, minus_logit,
            )

    score_map.flush()
    score_meta["completed_tiles"] = len(train_tiles)
    score_meta["complete"] = True
    atomic_json(SCORE_META_PATH, score_meta)

    del score_model
    gc.collect()

score_meta = json.loads(SCORE_META_PATH.read_text(encoding="utf-8"))
assert score_meta["complete"] is True
assert int(score_meta["completed_tiles"]) == len(train_tiles)

del score_map
gc.collect()

print("TRAIN-ONLY EXP045 SCORE CACHE: COMPLETE")


train score tiles 1,000/168,611 (0.59%) tiles/s=45.10 ETA=1.03h
train score tiles 2,000/168,611 (1.19%) tiles/s=44.96 ETA=1.03h
train score tiles 3,000/168,611 (1.78%) tiles/s=44.91 ETA=1.02h
train score tiles 4,000/168,611 (2.37%) tiles/s=44.90 ETA=1.02h
train score tiles 5,000/168,611 (2.97%) tiles/s=44.86 ETA=1.01h
train score tiles 6,000/168,611 (3.56%) tiles/s=45.00 ETA=1.00h
train score tiles 7,000/168,611 (4.15%) tiles/s=45.07 ETA=1.00h
train score tiles 8,000/168,611 (4.74%) tiles/s=45.15 ETA=0.99h
train score tiles 9,000/168,611 (5.34%) tiles/s=45.19 ETA=0.98h
train score tiles 10,000/168,611 (5.93%) tiles/s=45.22 ETA=0.97h
train score tiles 11,000/168,611 (6.52%) tiles/s=45.24 ETA=0.97h
train score tiles 12,000/168,611 (7.12%) tiles/s=45.28 ETA=0.96h
train score tiles 13,000/168,611 (7.71%) tiles/s=45.25 ETA=0.96h
train score tiles 14,000/168,611 (8.30%) tiles/s=45.19 ETA=0.95h
train score tiles 15,000/168,611 (8.90%) tiles/s=45.21 ETA=0.94h
train score tiles 16,000/168,611 (

## 4. Build exact 4-stratum EXP047 risk cache

Первый CPU-pass строит histogram только по valid train negatives.

Второй CPU-pass присваивает:
- stratum 0 = top 1%;
- stratum 1 = 1–5%;
- stratum 2 = 5–15%;
- stratum 3 = bottom 85%.

При ties используем deterministic stable order:
`tile_id → strand → base`.


In [5]:
if RISK_MANIFEST.is_file() and RISK_CACHE.is_file():
    risk_manifest = json.loads(RISK_MANIFEST.read_text(encoding="utf-8"))
    assert risk_manifest["source_checkpoint_sha256"] == base_sha
    assert tuple(risk_manifest["shape"]) == SCORE_SHAPE
    assert risk_manifest["complete"] is True
    expected_risk_bytes = int(np.prod(SCORE_SHAPE, dtype=np.int64))
    assert RISK_CACHE.stat().st_size == expected_risk_bytes
    actual_risk_sha = sha256_file(RISK_CACHE)
    assert risk_manifest["cache_sha256"] == actual_risk_sha
    print("Existing risk cache verified:", RISK_CACHE)

else:
    if RISK_CACHE.exists() or RISK_MANIFEST.exists():
        raise RuntimeError(
            "Incomplete risk cache state exists. "
            "Inspect/remove only the incomplete STEP04 risk artifacts."
        )

    score_map = np.memmap(
        SCORE_BIN_PATH,
        mode="r",
        dtype=np.uint32,
        shape=SCORE_SHAPE,
    )

    CPU_BATCH_TILES = 32
    histogram = np.zeros(SCORE_LEVELS, dtype=np.int64)

    started_hist = time.perf_counter()

    for start in range(0, len(train_tiles), CPU_BATCH_TILES):
        end = min(start + CPU_BATCH_TILES, len(train_tiles))
        selected = train_tiles.iloc[start:end]
        batch = train_generator.make_batch(selected)
        ids = batch.tile_id.numpy().astype(np.int64, copy=False)

        valid = batch.mask.numpy().astype(bool, copy=False)
        positive = batch.target.numpy().astype(bool, copy=False)
        negative = valid & ~positive

        bins = np.asarray(score_map[ids])
        histogram += np.bincount(
            bins[negative].astype(np.int64, copy=False),
            minlength=SCORE_LEVELS,
        )

        if end % 5000 < CPU_BATCH_TILES or end == len(train_tiles):
            print(
                f"histogram tiles {end:,}/{len(train_tiles):,}",
                flush=True,
            )
        del batch, valid, positive, negative, bins

    total_negative = int(histogram.sum())
    assert total_negative == int(train_generator.total_negative), (
        total_negative,
        train_generator.total_negative,
    )

    np.save(HISTOGRAM_PATH, histogram)

    cut_counts = np.asarray(
        [
            round(total_negative * 0.01),
            round(total_negative * 0.05),
            round(total_negative * 0.15),
        ],
        dtype=np.int64,
    )

    higher = (
        np.cumsum(histogram[::-1], dtype=np.int64)[::-1]
        - histogram
    )
    seen_by_score = np.zeros(SCORE_LEVELS, dtype=np.int64)

    temp_risk = RISK_CACHE.with_suffix(".dat.tmp")
    risk_map = np.memmap(
        temp_risk,
        mode="w+",
        dtype=np.uint8,
        shape=SCORE_SHAPE,
    )
    risk_map[:] = np.uint8(255)

    realized = np.zeros(4, dtype=np.int64)
    assigned = 0

    started_assign = time.perf_counter()

    for start in range(0, len(train_tiles), CPU_BATCH_TILES):
        end = min(start + CPU_BATCH_TILES, len(train_tiles))
        selected = train_tiles.iloc[start:end]
        batch = train_generator.make_batch(selected)
        ids = batch.tile_id.numpy().astype(np.int64, copy=False)

        valid = batch.mask.numpy().astype(bool, copy=False)
        positive = batch.target.numpy().astype(bool, copy=False)
        negative = valid & ~positive

        bins = np.asarray(score_map[ids])
        strata = np.full(bins.shape, 255, dtype=np.uint8)

        negative_scores = bins[negative].astype(np.int64, copy=False)
        order = np.argsort(negative_scores, kind="stable")
        ordered_scores = negative_scores[order]

        if len(ordered_scores):
            group_starts = np.r_[
                np.int64(0),
                np.flatnonzero(
                    ordered_scores[1:] != ordered_scores[:-1]
                ).astype(np.int64) + 1,
            ]
            group_ends = np.r_[
                group_starts[1:],
                np.int64(len(ordered_scores)),
            ]
            group_lengths = group_ends - group_starts
            unique_scores = ordered_scores[group_starts]

            offsets = (
                np.arange(len(ordered_scores), dtype=np.int64)
                - np.repeat(group_starts, group_lengths)
            )
            rank_bases = np.repeat(
                higher[unique_scores] + seen_by_score[unique_scores],
                group_lengths,
            )
            ordered_ranks = rank_bases + offsets

            ordered_strata = np.searchsorted(
                cut_counts,
                ordered_ranks,
                side="right",
            ).astype(np.uint8)

            stratum_values = np.empty(
                len(ordered_strata),
                dtype=np.uint8,
            )
            stratum_values[order] = ordered_strata
            strata[negative] = stratum_values

            seen_by_score[unique_scores] += group_lengths
            realized += np.bincount(
                ordered_strata,
                minlength=4,
            )
            assigned += len(ordered_strata)

        risk_map[ids] = strata

        if end % 5000 < CPU_BATCH_TILES or end == len(train_tiles):
            print(
                f"risk assignment tiles {end:,}/{len(train_tiles):,}",
                flush=True,
            )

        del (
            batch, valid, positive, negative,
            bins, strata, negative_scores, order, ordered_scores,
        )

    risk_map.flush()
    del risk_map, score_map
    gc.collect()

    assert assigned == total_negative
    assert int(realized.sum()) == total_negative

    os.replace(temp_risk, RISK_CACHE)

    fractions = realized.astype(np.float64) / total_negative
    multipliers = (
        0.5
        + 0.5
        * TARGET_NEGATIVE_STRATUM_FRACTION
        / fractions
    )

    expected_counts = np.asarray(
        [
            cut_counts[0],
            cut_counts[1] - cut_counts[0],
            cut_counts[2] - cut_counts[1],
            total_negative - cut_counts[2],
        ],
        dtype=np.int64,
    )
    assert np.array_equal(realized, expected_counts)

    risk_manifest = {
        "complete": True,
        "created": date.today().isoformat(),
        "assembly": ASSEMBLY,
        "source_checkpoint": str(BASE_CHECKPOINT),
        "source_checkpoint_sha256": base_sha,
        "source_score_cache": str(SCORE_BIN_PATH),
        "score_definition": "sigmoid(EXP045 logit), rounded to 5 decimals",
        "tie_break": "stable tile_id/strand/base order within equal score",
        "negative_strata": [
            "top_0_1pct",
            "top_1_5pct",
            "top_5_15pct",
            "bottom_85pct",
        ],
        "shape": list(SCORE_SHAPE),
        "dtype": "uint8",
        "invalid_or_positive_code": 255,
        "rank_cut_counts": cut_counts.tolist(),
        "stratum_counts": realized.tolist(),
        "natural_fractions": fractions.tolist(),
        "target_negative_fraction_each_stratum": 0.25,
        "candidate_negative_weight_multipliers": multipliers.tolist(),
        "total_train_negatives": total_negative,
        "cache_sha256": sha256_file(RISK_CACHE),
        "histogram_seconds": time.perf_counter() - started_hist,
        "assignment_seconds": time.perf_counter() - started_assign,
        "validation_used_for_mining": False,
    }
    atomic_json(RISK_MANIFEST, risk_manifest)

display(
    pd.DataFrame(
        {
            "stratum": risk_manifest["negative_strata"],
            "count": risk_manifest["stratum_counts"],
            "natural_fraction": risk_manifest["natural_fractions"],
            "loss_multiplier": risk_manifest[
                "candidate_negative_weight_multipliers"
            ],
        }
    )
)

print("Risk cache:", RISK_CACHE)
print("Risk SHA:", risk_manifest["cache_sha256"])
print("Validation used for mining:", risk_manifest["validation_used_for_mining"])


histogram tiles 5,024/168,611
histogram tiles 10,016/168,611
histogram tiles 15,008/168,611
histogram tiles 20,000/168,611
histogram tiles 25,024/168,611
histogram tiles 30,016/168,611
histogram tiles 35,008/168,611
histogram tiles 40,000/168,611
histogram tiles 45,024/168,611
histogram tiles 50,016/168,611
histogram tiles 55,008/168,611
histogram tiles 60,000/168,611
histogram tiles 65,024/168,611
histogram tiles 70,016/168,611
histogram tiles 75,008/168,611
histogram tiles 80,000/168,611
histogram tiles 85,024/168,611
histogram tiles 90,016/168,611
histogram tiles 95,008/168,611
histogram tiles 100,000/168,611
histogram tiles 105,024/168,611
histogram tiles 110,016/168,611
histogram tiles 115,008/168,611
histogram tiles 120,000/168,611
histogram tiles 125,024/168,611
histogram tiles 130,016/168,611
histogram tiles 135,008/168,611
histogram tiles 140,000/168,611
histogram tiles 145,024/168,611
histogram tiles 150,016/168,611
histogram tiles 155,008/168,611
histogram tiles 160,000/168,

,stratum,count,natural_fraction,loss_multiplier
0,top_0_1pct,3285350,0.01,13.000001
1,top_1_5pct,13141401,0.04,3.625000
2,top_5_15pct,32853503,0.10,1.750000
3,bottom_85pct,279254772,0.85,0.647059


Risk cache: D:\Projects\EPIC\EPIC\artifacts\experiments\SUBMIT_ilPloInte3p2_EXP047B_RISK_CONTINUATION\run_001\risk_strata_cache\train_negative_stratum_u8.dat
Risk SHA: 7bb048ea4ee639332371a6e0321a883d87834139f874cbe9d8ccf65b8bb224fb
Validation used for mining: False


## 5. One-step branch-B smoke

Это отдельные model/generator/optimizer, поэтому smoke не расходует RNG
реального continuation.


In [6]:
risk_manifest = json.loads(RISK_MANIFEST.read_text(encoding="utf-8"))
risk_hash = sha256_file(RISK_MANIFEST)

smoke_cache = np.memmap(
    RISK_CACHE,
    mode="r",
    dtype=np.uint8,
    shape=tuple(risk_manifest["shape"]),
)
risk_multipliers = np.asarray(
    risk_manifest["candidate_negative_weight_multipliers"],
    dtype=np.float32,
)

smoke_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
smoke_generator.rng.bit_generator.state = copy.deepcopy(
    base["generator_rng_state"]
)

torch.set_rng_state(base["torch_rng_state"])

device = torch_directml.device()
smoke_model = create_cnn_presence_intensity(channels=512)
smoke_model.load_state_dict(base["model_state_dict"])
smoke_model = smoke_model.to(device)

smoke_optimizer = DirectMLAdam(
    smoke_model.parameters(),
    lr=planned_lr(1),
)
smoke_optimizer.load_state_dict(base["optimizer_state_dict"])
for group in smoke_optimizer.param_groups:
    group["lr"] = planned_lr(1)

smoke_model.train()
batch = smoke_generator.sample_batch(BATCH_SIZE)

strata = np.asarray(
    smoke_cache[
        batch.tile_id.numpy().astype(np.int64, copy=False)
    ]
)
valid_negative = (
    batch.mask.numpy()
    & ~batch.target.numpy().astype(bool)
)
assert not np.any(strata[valid_negative] == 255)

multiplier = np.ones(strata.shape, dtype=np.float32)
for stratum, value in enumerate(risk_multipliers):
    multiplier[strata == stratum] = value

plus_weight = (
    batch.plus_loss_weight
    * torch.from_numpy(multiplier[:, 0:1, :])
)
minus_weight = (
    batch.minus_loss_weight_reverse
    * torch.from_numpy(multiplier[:, 1:2, ::-1].copy())
)

pc = batch.count[:, 0:1, :].contiguous()
mc = batch.count[:, 1:2, :].flip(-1).contiguous()

smoke_optimizer.zero_grad(set_to_none=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    pp, mp, pi, mi = forward_both_strands_multitask(
        smoke_model,
        batch.x_both_strands.to(device),
        batch_size=BATCH_SIZE,
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = dml_weighted_profile_bce_with_logits(
        pp, mp,
        batch.plus_target.to(device),
        batch.minus_target_reverse.to(device),
        plus_weight.to(device),
        minus_weight.to(device),
    )

    intensity_loss = (
        dml_weighted_huber(
            pi,
            torch.log1p(pc).to(device),
            (
                2.0
                * batch.plus_loss_weight
                * (pc > 0).float()
            ).to(device),
            delta=1.0,
        )
        + dml_weighted_huber(
            mi,
            torch.log1p(mc).to(device),
            (
                2.0
                * batch.minus_loss_weight_reverse
                * (mc > 0).float()
            ).to(device),
            delta=1.0,
        )
    )

    total_loss = (
        presence_loss
        + INTENSITY_LOSS_WEIGHT * intensity_loss
    )
    values = [
        float(x.detach().cpu().item())
        for x in (
            presence_loss,
            intensity_loss,
            total_loss,
        )
    ]
    assert all(math.isfinite(x) for x in values)

    total_loss.backward()
    smoke_optimizer.step()

fallbacks = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]
assert not fallbacks, fallbacks

print(
    "BRANCH-B SMOKE: PASS",
    "presence=", values[0],
    "intensity=", values[1],
    "total=", values[2],
)

del (
    smoke_cache, smoke_generator, smoke_model, smoke_optimizer,
    batch, strata, valid_negative, multiplier,
    plus_weight, minus_weight, pc, mc,
    pp, mp, pi, mi,
    presence_loss, intensity_loss, total_loss,
)
gc.collect()


BRANCH-B SMOKE: PASS presence= 0.24783915281295776 intensity= 0.18838679790496826 total= 0.2666778266429901


124

## 6. EXP047-B continuation: global 50k → 70k

Resume-safe:
- `latest.pt` каждые 500 local steps;
- immutable snapshots: 2k / 5k / 10k / 15k / 20k.

LR полностью совпадает с принятой EXP047-B continuation:
`3e-5 → warmup 1e-4 @500 → cosine 1e-5 @20k`.


In [7]:
def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }

def tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {k: tree_to_cpu(v) for k, v in value.items()}
    if isinstance(value, list):
        return [tree_to_cpu(v) for v in value]
    if isinstance(value, tuple):
        return tuple(tree_to_cpu(v) for v in value)
    return value

def continuation_payload(
    *,
    local_step,
    model,
    optimizer,
    generator,
    history,
    elapsed_seconds,
    seen_position_strand,
):
    return {
        "experiment": EXPERIMENT_ID,
        "branch": "B",
        "assembly": ASSEMBLY,
        "model_type": "presence_intensity",
        "architecture": "rf1029_width512",
        "base_experiment": "SUBMIT-MOTH-EXP045",
        "base_checkpoint": str(BASE_CHECKPOINT),
        "base_checkpoint_sha256": base_sha,
        "base_step": BASE_STEP,
        "local_step": int(local_step),
        "step": BASE_STEP + int(local_step),
        "model_state_dict": model_state_on_cpu(model),
        "optimizer_state_dict": tree_to_cpu(
            optimizer.state_dict()
        ),
        "generator_rng_state": copy.deepcopy(
            generator.rng.bit_generator.state
        ),
        "torch_rng_state": torch.get_rng_state(),
        "model_config": {
            "input_channels": 5,
            "channels": 512,
            "output_channels": 2,
            "theoretical_rf": 1029,
        },
        "profile_config": {
            "target_length": profile_config.target_length,
            "context_flank": profile_config.context_flank,
            "positive_branch_fraction": (
                profile_config.positive_branch_fraction
            ),
        },
        "training_config": {
            "continuation_steps": CONTINUATION_STEPS,
            "batch_size": BATCH_SIZE,
            "warmup_steps": WARMUP_STEPS,
            "start_lr": START_LR,
            "peak_lr": PEAK_LR,
            "end_lr": END_LR,
            "intensity_loss_weight": INTENSITY_LOSS_WEIGHT,
            "presence_objective": (
                "half_natural_half_equal_danger_strata_negative_BCE"
            ),
        },
        "risk_manifest_sha256": risk_hash,
        "seen_position_strand": int(seen_position_strand),
        "elapsed_seconds": float(elapsed_seconds),
        "history": list(history),
    }

if FINAL_CHECKPOINT.is_file():
    done = torch.load(
        FINAL_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )
    assert done["experiment"] == EXPERIMENT_ID
    assert done["branch"] == "B"
    assert done["assembly"] == ASSEMBLY
    assert int(done["local_step"]) == CONTINUATION_STEPS
    assert int(done["step"]) == 70_000
    assert done["base_checkpoint_sha256"] == base_sha
    assert done["risk_manifest_sha256"] == risk_hash
    print("EXP047-B already complete:", FINAL_CHECKPOINT)
    del done

else:
    current_risk_manifest = json.loads(
        RISK_MANIFEST.read_text(encoding="utf-8")
    )
    assert (
        current_risk_manifest["source_checkpoint_sha256"]
        == base_sha
    )
    assert (
        current_risk_manifest["cache_sha256"]
        == sha256_file(RISK_CACHE)
    )

    risk_cache = np.memmap(
        RISK_CACHE,
        mode="r",
        dtype=np.uint8,
        shape=tuple(current_risk_manifest["shape"]),
    )
    risk_multipliers = np.asarray(
        current_risk_manifest[
            "candidate_negative_weight_multipliers"
        ],
        dtype=np.float32,
    )

    generator = ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=42,
    )

    model = create_cnn_presence_intensity(channels=512)

    if LATEST_CHECKPOINT.is_file():
        resume = torch.load(
            LATEST_CHECKPOINT,
            map_location="cpu",
            weights_only=False,
        )
        assert resume["experiment"] == EXPERIMENT_ID
        assert resume["branch"] == "B"
        assert resume["assembly"] == ASSEMBLY
        assert resume["base_checkpoint_sha256"] == base_sha
        assert resume["risk_manifest_sha256"] == risk_hash

        model.load_state_dict(resume["model_state_dict"])
        generator.rng.bit_generator.state = copy.deepcopy(
            resume["generator_rng_state"]
        )
        torch.set_rng_state(resume["torch_rng_state"])

        start_step = int(resume["local_step"])
        history = list(resume["history"])
        seen = int(resume["seen_position_strand"])
        previous_elapsed = float(resume["elapsed_seconds"])

        print("RESUME EXP047-B at local step:", start_step)

    else:
        model.load_state_dict(base["model_state_dict"])
        generator.rng.bit_generator.state = copy.deepcopy(
            base["generator_rng_state"]
        )
        torch.set_rng_state(base["torch_rng_state"])

        start_step = 0
        history = []
        seen = 0
        previous_elapsed = 0.0
        resume = None

        print("Fresh EXP047-B continuation from global step 50k")

    device = torch_directml.device()
    model = model.to(device)

    optimizer = DirectMLAdam(
        model.parameters(),
        lr=START_LR,
    )

    if LATEST_CHECKPOINT.is_file():
        optimizer.load_state_dict(
            resume["optimizer_state_dict"]
        )
        del resume
    else:
        optimizer.load_state_dict(
            base["optimizer_state_dict"]
        )

    model.train()

    ema_presence = (
        history[-1]["ema_presence"]
        if history else None
    )
    ema_intensity = (
        history[-1]["ema_intensity"]
        if history else None
    )
    ema_total = (
        history[-1]["ema_total"]
        if history else None
    )

    started = time.perf_counter()

    for local_step in range(
        start_step + 1,
        CONTINUATION_STEPS + 1,
    ):
        lr = planned_lr(local_step)
        for group in optimizer.param_groups:
            group["lr"] = lr

        batch = generator.sample_batch(BATCH_SIZE)

        ids = batch.tile_id.numpy().astype(
            np.int64,
            copy=False,
        )
        strata = np.asarray(risk_cache[ids])

        valid_negative = (
            batch.mask.numpy()
            & ~batch.target.numpy().astype(bool)
        )
        if np.any(strata[valid_negative] == 255):
            raise AssertionError(
                "Risk cache misses sampled valid negative"
            )

        multiplier = np.ones(
            strata.shape,
            dtype=np.float32,
        )
        for stratum, value in enumerate(risk_multipliers):
            multiplier[strata == stratum] = value

        plus_weight = (
            batch.plus_loss_weight
            * torch.from_numpy(multiplier[:, 0:1, :])
        )
        minus_weight_reverse = (
            batch.minus_loss_weight_reverse
            * torch.from_numpy(
                multiplier[:, 1:2, ::-1].copy()
            )
        )

        x_gpu = batch.x_both_strands.to(device)
        plus_target_gpu = batch.plus_target.to(device)
        minus_target_gpu = (
            batch.minus_target_reverse.to(device)
        )
        plus_weight_gpu = plus_weight.to(device)
        minus_weight_gpu = minus_weight_reverse.to(device)

        plus_count = batch.count[:, 0:1, :].contiguous()
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target = torch.log1p(
            plus_count
        ).to(device)
        minus_intensity_target = torch.log1p(
            minus_count_reverse
        ).to(device)

        plus_intensity_weight = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)
        minus_intensity_weight = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        optimizer.zero_grad(set_to_none=True)

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            (
                plus_presence,
                minus_presence,
                plus_intensity,
                minus_intensity,
            ) = forward_both_strands_multitask(
                model,
                x_gpu,
                batch_size=BATCH_SIZE,
                context_flank=profile_config.context_flank,
                target_length=profile_config.target_length,
            )

            presence_loss = (
                dml_weighted_profile_bce_with_logits(
                    plus_presence,
                    minus_presence,
                    plus_target_gpu,
                    minus_target_gpu,
                    plus_weight_gpu,
                    minus_weight_gpu,
                )
            )

            intensity_loss = (
                dml_weighted_huber(
                    plus_intensity,
                    plus_intensity_target,
                    plus_intensity_weight,
                    delta=1.0,
                )
                + dml_weighted_huber(
                    minus_intensity,
                    minus_intensity_target,
                    minus_intensity_weight,
                    delta=1.0,
                )
            )

            total_loss = (
                presence_loss
                + INTENSITY_LOSS_WEIGHT
                * intensity_loss
            )

            values = [
                float(x.detach().cpu().item())
                for x in (
                    presence_loss,
                    intensity_loss,
                    total_loss,
                )
            ]
            if not all(math.isfinite(x) for x in values):
                raise FloatingPointError(
                    f"Non-finite loss at local step "
                    f"{local_step}: {values}"
                )

            total_loss.backward()
            optimizer.step()

        new_fallback = [
            str(item.message)
            for item in caught
            if "fall back" in str(item.message).lower()
        ]
        if new_fallback:
            raise RuntimeError(
                "DirectML CPU fallback: "
                + " | ".join(new_fallback)
            )

        pv, iv, tv = values

        if ema_presence is None:
            ema_presence, ema_intensity, ema_total = values
        else:
            ema_presence = (
                0.99 * ema_presence + 0.01 * pv
            )
            ema_intensity = (
                0.99 * ema_intensity + 0.01 * iv
            )
            ema_total = (
                0.99 * ema_total + 0.01 * tv
            )

        seen += int(batch.mask.sum())

        elapsed = (
            previous_elapsed
            + time.perf_counter()
            - started
        )

        if (
            local_step % LOG_EVERY == 0
            or local_step == CONTINUATION_STEPS
        ):
            run_elapsed = max(
                elapsed - previous_elapsed,
                1e-9,
            )
            completed_now = local_step - start_step
            sec_per_step = (
                run_elapsed / completed_now
            )
            record = {
                "branch": "B",
                "local_step": local_step,
                "step": BASE_STEP + local_step,
                "learning_rate": lr,
                "presence_loss": pv,
                "intensity_loss": iv,
                "total_loss": tv,
                "ema_presence": float(ema_presence),
                "ema_intensity": float(ema_intensity),
                "ema_total": float(ema_total),
                "elapsed_seconds": float(elapsed),
                "seconds_per_step_current_process": float(
                    sec_per_step
                ),
                "estimated_hours_remaining": float(
                    (CONTINUATION_STEPS - local_step)
                    * sec_per_step
                    / 3600
                ),
                "seen_position_strand": int(seen),
            }
            history.append(record)
            pd.DataFrame(history).to_csv(
                TRAINING_HISTORY,
                index=False,
            )
            print(
                f"[B] local={local_step:05d}/"
                f"{CONTINUATION_STEPS} "
                f"global={BASE_STEP + local_step} "
                f"lr={lr:.7g} "
                f"loss=({pv:.5f},{iv:.5f},{tv:.5f}) "
                f"sec/step={sec_per_step:.3f} "
                f"ETA={record['estimated_hours_remaining']:.2f}h",
                flush=True,
            )

        if (
            local_step % LATEST_EVERY == 0
            or local_step == CONTINUATION_STEPS
        ):
            payload = continuation_payload(
                local_step=local_step,
                model=model,
                optimizer=optimizer,
                generator=generator,
                history=history,
                elapsed_seconds=elapsed,
                seen_position_strand=seen,
            )
            atomic_torch_save(
                LATEST_CHECKPOINT,
                payload,
            )
            if local_step in SNAPSHOT_STEPS:
                atomic_torch_save(
                    BRANCH_DIR
                    / f"model_local_step_{local_step:05d}.pt",
                    payload,
                )
            if local_step == CONTINUATION_STEPS:
                atomic_torch_save(
                    FINAL_CHECKPOINT,
                    payload,
                )
            del payload

        del (
            batch, ids, strata, valid_negative,
            multiplier, plus_weight,
            minus_weight_reverse,
            x_gpu, plus_target_gpu,
            minus_target_gpu,
            plus_weight_gpu,
            minus_weight_gpu,
            plus_count,
            minus_count_reverse,
            plus_intensity_target,
            minus_intensity_target,
            plus_intensity_weight,
            minus_intensity_weight,
            plus_presence,
            minus_presence,
            plus_intensity,
            minus_intensity,
            presence_loss,
            intensity_loss,
            total_loss,
        )

    assert FINAL_CHECKPOINT.is_file()

    del (
        risk_cache,
        model,
        optimizer,
        generator,
    )
    gc.collect()

    print("EXP047-B TRAINING COMPLETE:", FINAL_CHECKPOINT)


Fresh EXP047-B continuation from global step 50k
[B] local=00250/20000 global=50250 lr=6.5e-05 loss=(0.24057,0.37709,0.27828) sec/step=0.233 ETA=1.28h
[B] local=00500/20000 global=50500 lr=0.0001 loss=(0.49368,0.14411,0.50809) sec/step=0.231 ETA=1.25h
[B] local=00750/20000 global=50750 lr=9.99635e-05 loss=(0.32862,0.17021,0.34564) sec/step=0.230 ETA=1.23h
[B] local=01000/20000 global=51000 lr=9.985408e-05 loss=(0.41714,0.25518,0.44266) sec/step=0.230 ETA=1.21h
[B] local=01250/20000 global=51250 lr=9.96719e-05 loss=(0.37143,0.21112,0.39254) sec/step=0.230 ETA=1.20h
[B] local=01500/20000 global=51500 lr=9.941726e-05 loss=(0.59863,0.19509,0.61813) sec/step=0.230 ETA=1.18h
[B] local=01750/20000 global=51750 lr=9.909058e-05 loss=(0.48686,0.08863,0.49572) sec/step=0.230 ETA=1.17h
[B] local=02000/20000 global=52000 lr=9.869238e-05 loss=(0.51389,0.03492,0.51738) sec/step=0.230 ETA=1.15h
[B] local=02250/20000 global=52250 lr=9.822332e-05 loss=(0.43415,0.31909,0.46606) sec/step=0.230 ETA=1.13h
[

## 7. Final checkpoint integrity


In [8]:
final = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

assert final["experiment"] == EXPERIMENT_ID
assert final["branch"] == "B"
assert final["assembly"] == ASSEMBLY
assert int(final["base_step"]) == 50_000
assert int(final["local_step"]) == 20_000
assert int(final["step"]) == 70_000
assert final["base_checkpoint_sha256"] == base_sha
assert final["risk_manifest_sha256"] == risk_hash

final_sha = sha256_file(FINAL_CHECKPOINT)

print("Final global step:", final["step"])
print("Final SHA256:", final_sha)
print(
    "Continuation hours:",
    round(final["elapsed_seconds"] / 3600, 3),
)


Final global step: 70000
Final SHA256: 975adb5184bff1ef5c760db94d69ac016e9b55084a3a0dfefab2a8e9d32cfed5
Continuation hours: 1.274


## 8. One full natural validation scan at global step 70k

Это первая validation после continuation. Она ничего не меняет в training.


In [9]:
device = torch_directml.device()

validation_model = create_cnn_presence_intensity(channels=512)
validation_model.load_state_dict(final["model_state_dict"])
validation_model = validation_model.to(device)
validation_model.eval()

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

validation_result = evaluate_presence_intensity_model(
    validation_model,
    validation_generator,
    device=device,
    batch_size=1,
    score_decimals=5,
    progress_every_batches=100,
    progress=print,
    fail_on_cpu_fallback=True,
)

display(validation_result.metrics)

validation_result.metrics.to_csv(
    VALIDATION_METRICS,
    index=False,
)

assert (
    validation_result.processed_positions
    == 104_114_200
)
assert (
    validation_result.processed_positives
    == 162_306
)
assert not validation_result.fallback_warnings

del validation_model, validation_generator
gc.collect()


tiles=     1/53710 positions=      634 positions/s=26,046
tiles=   100/53710 positions=   184526 positions/s=78,317
tiles=   200/53710 positions=   381312 positions/s=81,854
tiles=   300/53710 positions=   579586 positions/s=83,185
tiles=   400/53710 positions=   774616 positions/s=83,532
tiles=   500/53710 positions=   965744 positions/s=83,368
tiles=   600/53710 positions=  1160024 positions/s=83,505
tiles=   700/53710 positions=  1356748 positions/s=83,685
tiles=   800/53710 positions=  1545988 positions/s=83,499
tiles=   900/53710 positions=  1729542 positions/s=83,092
tiles=  1000/53710 positions=  1916654 positions/s=82,742
tiles=  1100/53710 positions=  2110016 positions/s=82,849
tiles=  1200/53710 positions=  2298690 positions/s=82,727
tiles=  1300/53710 positions=  2484916 positions/s=82,598
tiles=  1400/53710 positions=  2677334 positions/s=82,434
tiles=  1500/53710 positions=  2874022 positions/s=82,597
tiles=  1600/53710 positions=  3068758 positions/s=82,774
tiles=  1700/5

,score,segment,position_strand,positives,prevalence,average_precision,ap_lift,epic_spearman,quantized_spearman
0,presence_probability,overall,104114200,162306,0.001559,0.170004,109.051957,0.169896,0.154801
1,presence_probability,NC_071299.1,22387740,31959,0.001428,0.168496,118.033676,0.228773,0.222676
2,presence_probability,NC_071301.1,21156176,30177,0.001426,0.188225,131.959104,0.218956,0.212939
3,presence_probability,NC_071314.1,15631794,31839,0.002037,0.210814,103.502076,0.244989,0.234851
4,presence_probability,NC_071316.1,13373548,16558,0.001238,0.108165,87.362978,0.181458,0.180337
5,presence_probability,NC_071317.1,13344206,20148,0.001510,0.164288,108.809536,0.225903,0.221749
6,presence_probability,NC_071320.1,9555594,17469,0.001828,0.146770,80.283592,0.183476,0.180897
7,presence_probability,NC_071321.1,8665142,14156,0.001634,0.161115,98.621096,0.248056,0.245057
8,intensity_prediction,overall,104114200,162306,0.001559,0.093023,59.671526,0.207417,0.296105
9,intensity_prediction,NC_071299.1,22387740,31959,0.001428,0.089913,62.985395,0.267215,0.324168


401

## 9. EXP045 @50k vs fixed EXP047-B @70k

Это **отчёт**, а не checkpoint selection. Для последующего benchmark-submission
остаётся EXP047-B @70k.


In [10]:
base_metrics = pd.read_csv(BASE_METRICS)
b_metrics = pd.read_csv(VALIDATION_METRICS)

def presence_only(frame):
    if "score" in frame.columns:
        frame = frame.loc[
            frame["score"] == "presence_probability"
        ].copy()
    return frame

base_presence = presence_only(base_metrics)
b_presence = presence_only(b_metrics)

compare = base_presence.merge(
    b_presence,
    on="segment",
    suffixes=("_EXP045", "_EXP047B"),
    validate="one_to_one",
)

compare["delta_ap"] = (
    compare["average_precision_EXP047B"]
    - compare["average_precision_EXP045"]
)
compare["delta_spearman"] = (
    compare["epic_spearman_EXP047B"]
    - compare["epic_spearman_EXP045"]
)

display(
    compare[
        [
            "segment",
            "average_precision_EXP045",
            "average_precision_EXP047B",
            "delta_ap",
            "epic_spearman_EXP045",
            "epic_spearman_EXP047B",
            "delta_spearman",
        ]
    ]
)

overall = compare.loc[
    compare["segment"] == "overall"
].iloc[0]

result = {
    "experiment": EXPERIMENT_ID,
    "assembly": ASSEMBLY,
    "recipe": "fixed accepted EXP047-B",
    "base_checkpoint": str(BASE_CHECKPOINT),
    "base_checkpoint_sha256": base_sha,
    "final_checkpoint": str(FINAL_CHECKPOINT),
    "final_checkpoint_sha256": final_sha,
    "global_step": 70_000,
    "risk_manifest": str(RISK_MANIFEST),
    "risk_manifest_sha256": risk_hash,
    "validation_position_strand": int(
        validation_result.processed_positions
    ),
    "validation_positives": int(
        validation_result.processed_positives
    ),
    "exp045_ap": float(
        overall["average_precision_EXP045"]
    ),
    "exp047b_ap": float(
        overall["average_precision_EXP047B"]
    ),
    "delta_ap": float(overall["delta_ap"]),
    "exp045_spearman": float(
        overall["epic_spearman_EXP045"]
    ),
    "exp047b_spearman": float(
        overall["epic_spearman_EXP047B"]
    ),
    "delta_spearman": float(
        overall["delta_spearman"]
    ),
    "validation_used_for_risk_mining": False,
    "checkpoint_selected_on_validation": False,
    "next_stage": (
        "official ilPloInte3.2 test inference "
        "and submission writer"
    ),
}

atomic_json(RESULT_SUMMARY, result)

print(json.dumps(result, ensure_ascii=False, indent=2))
print()
print("RESULT:", RESULT_SUMMARY)
print("NEXT: official TEST inference + submission")


,segment,average_precision_EXP045,average_precision_EXP047B,delta_ap,epic_spearman_EXP045,epic_spearman_EXP047B,delta_spearman
0,overall,0.158191,0.170004,0.011812,0.151259,0.169896,0.018637
1,NC_071299.1,0.159201,0.168496,0.009294,0.213569,0.228773,0.015203
2,NC_071301.1,0.173290,0.188225,0.014935,0.200586,0.218956,0.018370
3,NC_071314.1,0.196656,0.210814,0.014158,0.225410,0.244989,0.019579
4,NC_071316.1,0.098372,0.108165,0.009793,0.163312,0.181458,0.018146
5,NC_071317.1,0.156300,0.164288,0.007989,0.208868,0.225903,0.017034
6,NC_071320.1,0.133498,0.146770,0.013272,0.170174,0.183476,0.013302
7,NC_071321.1,0.149838,0.161115,0.011277,0.236272,0.248056,0.011784


{
  "experiment": "SUBMIT-MOTH-EXP047B",
  "assembly": "ilPloInte3.2",
  "recipe": "fixed accepted EXP047-B",
  "base_checkpoint": "D:\\Projects\\EPIC\\EPIC\\artifacts\\experiments\\SUBMIT_ilPloInte3p2_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\\run_001\\model_step_50000.pt",
  "base_checkpoint_sha256": "01dd1761937dbe15ce334689db27221cc0801514a2bf9a7e63ee81ec9d521048",
  "final_checkpoint": "D:\\Projects\\EPIC\\EPIC\\artifacts\\experiments\\SUBMIT_ilPloInte3p2_EXP047B_RISK_CONTINUATION\\run_001\\branch_B\\branch_B_final.pt",
  "final_checkpoint_sha256": "975adb5184bff1ef5c760db94d69ac016e9b55084a3a0dfefab2a8e9d32cfed5",
  "global_step": 70000,
  "risk_manifest": "D:\\Projects\\EPIC\\EPIC\\artifacts\\experiments\\SUBMIT_ilPloInte3p2_EXP047B_RISK_CONTINUATION\\run_001\\risk_strata_cache\\risk_strata_manifest.json",
  "risk_manifest_sha256": "aed5a793f14d3b2be9c8e044eca6047a951a106e5c32a861e15abbe573998283",
  "validation_position_strand": 104114200,
  "validation_positives": 162306,
  "exp